<a href="https://colab.research.google.com/github/MO230101/Copolymer-lipid-interaction-study_ver.3/blob/main/04_Chemical_representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 04_Chemical_representation.py
# ============================================================
#
# PURPOSE
# -------
# Generate leakage-safe chemical representations from
# composition + molecular structure ONLY.
#
# INPUT
#   1) copolymer_composition.csv
#   2) component_smiles.csv
#
# OUTPUT
#   01_RDKit_Component_Descriptors.csv
#   02_RDKit_RoleSeparated_Raw.csv
#   03_RDKit_Numeric_Final.csv
#   04_Chemical_Language_Sentences.csv
#   05_Chemical_Language_Embedding_384D.csv
#   06_Chemical_representation_audit.csv
#   07_Chemical_representation_source_data.xlsx
#   08_Metadata.json
#
#   ZIP:
#   04_Chemical_representation_output.zip
#
# IMPORTANT
# ---------
# NO TD-NMR
# NO Solution-NMR
# NO Dynamic-State Language
# NO experimental response
#
# Chemical Language is generated ONLY from:
#   - building-block identity
#   - chemical structure
#   - role
#   - composition
#
# ============================================================


# ============================================================
# 0. INSTALL
# ============================================================

import sys
import subprocess
import importlib.util


def ensure_package(import_name, pip_name=None):

    if pip_name is None:
        pip_name = import_name

    if importlib.util.find_spec(import_name) is None:

        print(
            f"[INSTALL] {pip_name}"
        )

        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                pip_name,
            ]
        )


ensure_package(
    "rdkit",
    "rdkit"
)

ensure_package(
    "sentence_transformers",
    "sentence-transformers"
)

ensure_package(
    "openpyxl",
    "openpyxl"
)


# ============================================================
# 1. IMPORT
# ============================================================

import os
import re
import json
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski

from sentence_transformers import SentenceTransformer


# ============================================================
# 2. SETTINGS
# ============================================================

EXPECTED_N = 43

LANGUAGE_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

OUTPUT_DIR = Path(
    "04_Chemical_representation_output"
)

ZIP_PATH = Path(
    "04_Chemical_representation_output.zip"
)


if OUTPUT_DIR.exists():

    shutil.rmtree(
        OUTPUT_DIR
    )


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


if ZIP_PATH.exists():

    ZIP_PATH.unlink()


# ============================================================
# 3. FINAL RDKit DESCRIPTORS
# ============================================================
#
# Same chemically interpretable six-dimensional family used
# in the later analysis:
#
#   cLogP
#   TPSA
#   Hbond = HBD + HBA
#   RotatableBonds
#   IonicCharacter
#   HeteroatomFraction
#
# ============================================================

FINAL_DESCRIPTOR_NAMES = [

    "cLogP",

    "TPSA",

    "Hbond",

    "RotatableBonds",

    "IonicCharacter",

    "HeteroatomFraction",
]


# ============================================================
# 4. HELPERS
# ============================================================

def robust_read_csv(path):

    last_error = None

    for encoding in [

        "utf-8-sig",
        "utf-8",
        "cp932",
        "latin1",

    ]:

        try:

            df = pd.read_csv(
                path,
                encoding=encoding
            )

            print(
                f"[INFO] Loaded {path.name} "
                f"with encoding={encoding}"
            )

            return df

        except Exception as e:

            last_error = e

    raise last_error


def clean_string(x):

    if pd.isna(x):

        return ""

    return str(
        x
    ).strip()


def canonical_component_name(x):

    s = clean_string(
        x
    )

    s = re.sub(
        r"\s+",
        "",
        s
    )

    return s


def canonical_id(x):

    s = clean_string(
        x
    )

    s = s.replace(
        "-",
        "_"
    )

    s = re.sub(
        r"\s+",
        "",
        s
    )

    return s.upper()


def normalize_role(x):

    s = clean_string(
        x
    ).lower()

    if any(
        key in s
        for key in [
            "hydrophilic",
            "hydrophile",
            "hydrophilic_monomer",
        ]
    ):

        return "hydrophilic"

    if any(
        key in s
        for key in [
            "hydrophobic",
            "hydrophobe",
            "hydrophobic_monomer",
        ]
    ):

        return "hydrophobic"

    if any(
        key in s
        for key in [
            "crosslink",
            "cross-link",
            "crosslinker",
        ]
    ):

        return "crosslinker"

    return s


# ============================================================
# 5. UPLOAD INPUT FILES
# ============================================================

try:

    from google.colab import files

    print(
        "=" * 80
    )

    print(
        "Upload TWO files:"
    )

    print(
        "1) copolymer_composition.csv"
    )

    print(
        "2) component_smiles.csv"
    )

    print(
        "=" * 80
    )

    uploaded = files.upload()

    uploaded_paths = [

        Path(
            x
        )

        for x in uploaded.keys()
    ]

except ImportError:

    uploaded_paths = list(
        Path(".").glob("*.csv")
    )


# ============================================================
# 6. IDENTIFY INPUT FILES
# ============================================================

composition_candidates = [

    p

    for p in uploaded_paths

    if (
        "copolymer_composition"
        in p.name.lower()
    )
]


smiles_candidates = [

    p

    for p in uploaded_paths

    if (
        "component_smiles"
        in p.name.lower()
    )
]


if len(
    composition_candidates
) == 0:

    raise FileNotFoundError(
        "copolymer_composition.csv "
        "was not found."
    )


if len(
    smiles_candidates
) == 0:

    raise FileNotFoundError(
        "component_smiles.csv "
        "was not found."
    )


COMPOSITION_FILE = (
    composition_candidates[0]
)

SMILES_FILE = (
    smiles_candidates[0]
)


print(
    "\nComposition:",
    COMPOSITION_FILE
)

print(
    "SMILES:",
    SMILES_FILE
)


# ============================================================
# 7. LOAD
# ============================================================

comp = robust_read_csv(
    COMPOSITION_FILE
)

smiles_df = robust_read_csv(
    SMILES_FILE
)


comp.columns = [

    str(c).strip()

    for c in comp.columns
]


smiles_df.columns = [

    str(c).strip()

    for c in smiles_df.columns
]


print(
    "\nComposition columns:"
)

print(
    comp.columns.tolist()
)


print(
    "\nSMILES columns:"
)

print(
    smiles_df.columns.tolist()
)


# ============================================================
# 8. STANDARDIZE COMPOSITION COLUMN NAMES
# ============================================================

composition_aliases = {

    "Copolymer_Name": [
        "Copolymer_Name",
        "Canonical_ID",
        "Sample",
        "Sample_Name",
        "copolymer",
    ],

    "mono_1": [
        "mono_1",
        "monomer_1",
        "component_1",
    ],

    "mono_2": [
        "mono_2",
        "monomer_2",
        "component_2",
    ],

    "mono_3": [
        "mono_3",
        "monomer_3",
        "component_3",
    ],

    "comp_1": [
        "comp_1",
        "composition_1",
        "ratio_1",
        "fraction_1",
    ],

    "comp_2": [
        "comp_2",
        "composition_2",
        "ratio_2",
        "fraction_2",
    ],

    "comp_3": [
        "comp_3",
        "composition_3",
        "ratio_3",
        "fraction_3",
    ],
}


def find_alias(
    columns,
    aliases
):

    lookup = {

        str(c).lower():
            c

        for c in columns
    }

    for alias in aliases:

        if alias.lower() in lookup:

            return lookup[
                alias.lower()
            ]

    return None


rename_comp = {}


for target, aliases in (
    composition_aliases.items()
):

    source = find_alias(
        comp.columns,
        aliases
    )

    if source is None:

        raise ValueError(

            f"Could not identify "
            f"composition column: {target}"
        )

    rename_comp[
        source
    ] = target


comp = comp.rename(
    columns=rename_comp
)


# ============================================================
# 9. STANDARDIZE SMILES COLUMN NAMES
# ============================================================

smiles_aliases = {

    "Component": [
        "Component",
        "component",
        "Name",
        "name",
        "Monomer",
        "monomer",
    ],

    "SMILES": [
        "SMILES",
        "smiles",
        "Canonical_SMILES",
        "canonical_smiles",
    ],
}


rename_smiles = {}


for target, aliases in (
    smiles_aliases.items()
):

    source = find_alias(
        smiles_df.columns,
        aliases
    )

    if source is None:

        raise ValueError(

            f"Could not identify "
            f"SMILES column: {target}"
        )

    rename_smiles[
        source
    ] = target


smiles_df = smiles_df.rename(
    columns=rename_smiles
)


# ============================================================
# 10. CLEAN COMPOSITION TABLE
# ============================================================

comp[
    "Canonical_ID"
] = (

    comp[
        "Copolymer_Name"
    ]

    .apply(
        canonical_id
    )
)


for col in [

    "mono_1",
    "mono_2",
    "mono_3",

]:

    comp[
        col
    ] = (

        comp[
            col
        ]

        .apply(
            canonical_component_name
        )
    )


for col in [

    "comp_1",
    "comp_2",
    "comp_3",

]:

    comp[
        col
    ] = pd.to_numeric(

        comp[
            col
        ],

        errors="coerce"
    )


if comp[
    [
        "comp_1",
        "comp_2",
        "comp_3",
    ]
].isna().any().any():

    raise ValueError(
        "Composition contains "
        "non-numeric or missing values."
    )


comp[
    "Composition_sum"
] = (

    comp[
        [
            "comp_1",
            "comp_2",
            "comp_3",
        ]
    ]

    .sum(
        axis=1
    )
)


# ============================================================
# 11. ROLE ASSIGNMENT
# ============================================================
#
# Composition file definition:
#
# mono_1 = hydrophilic block
# mono_2 = hydrophobic block
# mono_3 = crosslinker
#
# ============================================================

comp[
    "Hydrophilic_component"
] = comp[
    "mono_1"
]


comp[
    "Hydrophobic_component"
] = comp[
    "mono_2"
]


comp[
    "Crosslinker_component"
] = comp[
    "mono_3"
]


comp[
    "Hydrophilic_fraction"
] = comp[
    "comp_1"
]


comp[
    "Hydrophobic_fraction"
] = comp[
    "comp_2"
]


comp[
    "Crosslinker_fraction"
] = comp[
    "comp_3"
]


# ============================================================
# 12. CLEAN SMILES TABLE
# ============================================================

smiles_df[
    "Component"
] = (

    smiles_df[
        "Component"
    ]

    .apply(
        canonical_component_name
    )
)


smiles_df[
    "SMILES"
] = (

    smiles_df[
        "SMILES"
    ]

    .astype(
        str
    )

    .str.strip()
)


if (
    smiles_df[
        "Component"
    ]
    .duplicated()
    .any()
):

    duplicates = (

        smiles_df.loc[
            smiles_df[
                "Component"
            ]
            .duplicated(
                keep=False
            )
        ]
    )

    raise ValueError(

        "Duplicate component names "
        "in component_smiles.csv:\n"

        + duplicates.to_string(
            index=False
        )
    )


# ============================================================
# 13. CHECK COMPONENT COVERAGE
# ============================================================

composition_components = set(

    pd.concat(
        [
            comp["mono_1"],
            comp["mono_2"],
            comp["mono_3"],
        ]
    )
    .dropna()
    .unique()
)


smiles_components = set(

    smiles_df[
        "Component"
    ]
    .dropna()
    .unique()
)


missing_components = sorted(

    composition_components
    -
    smiles_components
)


if missing_components:

    raise ValueError(

        "SMILES missing for components:\n"

        + "\n".join(
            missing_components
        )
    )


# ============================================================
# 14. RDKit DESCRIPTOR FUNCTIONS
# ============================================================

def ionic_character(
    mol
):

    formal_charge = sum(

        abs(
            atom.GetFormalCharge()
        )

        for atom in mol.GetAtoms()
    )

    charged_atoms = sum(

        1

        for atom in mol.GetAtoms()

        if atom.GetFormalCharge() != 0
    )

    # Binary/graded structural ionic-character descriptor.
    #
    # 0 = no formally charged atom
    # >0 = formally charged structure.
    #
    # This preserves charge information without using
    # experimental data.

    return float(
        max(
            formal_charge,
            charged_atoms
        )
    )


def heteroatom_fraction(
    mol
):

    heavy_atoms = mol.GetNumHeavyAtoms()

    if heavy_atoms == 0:

        return np.nan

    hetero_atoms = sum(

        1

        for atom in mol.GetAtoms()

        if atom.GetAtomicNum()
        not in [
            1,
            6,
        ]
    )

    return (
        hetero_atoms
        /
        heavy_atoms
    )


def calculate_rdkit_descriptors(
    smiles
):

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:

        raise ValueError(
            f"RDKit could not parse SMILES: "
            f"{smiles}"
        )

    hbd = Lipinski.NumHDonors(
        mol
    )

    hba = Lipinski.NumHAcceptors(
        mol
    )

    return {

        "cLogP":
            float(
                Descriptors.MolLogP(
                    mol
                )
            ),

        "TPSA":
            float(
                Descriptors.TPSA(
                    mol
                )
            ),

        "HBD":
            float(
                hbd
            ),

        "HBA":
            float(
                hba
            ),

        "Hbond":
            float(
                hbd
                +
                hba
            ),

        "RotatableBonds":
            float(
                Lipinski.NumRotatableBonds(
                    mol
                )
            ),

        "IonicCharacter":
            ionic_character(
                mol
            ),

        "HeteroatomFraction":
            heteroatom_fraction(
                mol
            ),
    }


# ============================================================
# 15. COMPONENT DESCRIPTORS
# ============================================================

descriptor_rows = []


for _, row in (
    smiles_df.iterrows()
):

    component = row[
        "Component"
    ]

    smiles = row[
        "SMILES"
    ]

    descriptors = (
        calculate_rdkit_descriptors(
            smiles
        )
    )

    rec = {

        "Component":
            component,

        "SMILES":
            smiles,
    }

    rec.update(
        descriptors
    )

    descriptor_rows.append(
        rec
    )


component_desc = pd.DataFrame(
    descriptor_rows
)


COMPONENT_PATH = (

    OUTPUT_DIR
    /
    "01_RDKit_Component_Descriptors.csv"
)


component_desc.to_csv(

    COMPONENT_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 16. COMPONENT DESCRIPTOR LOOKUP
# ============================================================

descriptor_lookup = {

    row[
        "Component"
    ]:
    row

    for _, row in (
        component_desc.iterrows()
    )
}


# ============================================================
# 17. ROLE-SEPARATED RDKit REPRESENTATION
# ============================================================
#
# Preserve hydrophilic / hydrophobic / crosslinker roles.
#
# Each descriptor is multiplied by the corresponding
# composition fraction.
#
# ============================================================

role_rows = []


ROLE_DEFINITIONS = [

    (
        "Hydrophilic",
        "mono_1",
        "comp_1",
    ),

    (
        "Hydrophobic",
        "mono_2",
        "comp_2",
    ),

    (
        "Crosslinker",
        "mono_3",
        "comp_3",
    ),
]


for _, row in (
    comp.iterrows()
):

    rec = {

        "Canonical_ID":
            row[
                "Canonical_ID"
            ],

        "Copolymer_Name":
            row[
                "Copolymer_Name"
            ],
    }


    for (
        role,
        component_col,
        fraction_col,
    ) in ROLE_DEFINITIONS:

        component = row[
            component_col
        ]

        fraction = float(
            row[
                fraction_col
            ]
        )

        desc = (
            descriptor_lookup[
                component
            ]
        )


        rec[
            f"{role}_Component"
        ] = component


        rec[
            f"{role}_Fraction"
        ] = fraction


        for descriptor in (
            FINAL_DESCRIPTOR_NAMES
        ):

            raw_value = float(
                desc[
                    descriptor
                ]
            )

            rec[
                f"{role}_{descriptor}_Raw"
            ] = raw_value

            rec[
                f"{role}_{descriptor}_Weighted"
            ] = (
                fraction
                *
                raw_value
            )


    role_rows.append(
        rec
    )


role_df = pd.DataFrame(
    role_rows
)


ROLE_PATH = (

    OUTPUT_DIR
    /
    "02_RDKit_RoleSeparated_Raw.csv"
)


role_df.to_csv(

    ROLE_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 18. FINAL RDKit NUMERIC REPRESENTATION
# ============================================================
#
# Final six dimensions:
#
# composition-weighted average of the three roles.
#
# This is Y-independent.
#
# ============================================================

rdkit_final = role_df[
    [
        "Canonical_ID",
        "Copolymer_Name",
    ]
].copy()


for descriptor in (
    FINAL_DESCRIPTOR_NAMES
):

    cols = [

        f"Hydrophilic_{descriptor}_Weighted",

        f"Hydrophobic_{descriptor}_Weighted",

        f"Crosslinker_{descriptor}_Weighted",
    ]


    rdkit_final[
        descriptor
    ] = (

        role_df[
            cols
        ]

        .sum(
            axis=1
        )
    )


RDKIT_FINAL_PATH = (

    OUTPUT_DIR
    /
    "03_RDKit_Numeric_Final.csv"
)


rdkit_final.to_csv(

    RDKIT_FINAL_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 19. CHEMICAL LANGUAGE HELPERS
# ============================================================

def fmt_fraction(
    x
):

    return (
        f"{float(x):.3f}"
    )


def descriptor_phrase(
    descriptor_name,
    value
):

    value = float(
        value
    )

    if descriptor_name == "cLogP":

        return (
            f"cLogP {value:.2f}"
        )

    if descriptor_name == "TPSA":

        return (
            f"TPSA {value:.1f} square angstrom"
        )

    if descriptor_name == "Hbond":

        return (
            f"{value:.0f} hydrogen-bond "
            f"donor/acceptor sites in total"
        )

    if descriptor_name == "RotatableBonds":

        return (
            f"{value:.0f} rotatable bonds"
        )

    if descriptor_name == "IonicCharacter":

        if value > 0:

            return (
                "formal ionic character"
            )

        return (
            "no formal ionic charge"
        )

    if descriptor_name == (
        "HeteroatomFraction"
    ):

        return (
            f"heteroatom fraction "
            f"{value:.2f}"
        )

    return (
        f"{descriptor_name} {value:.3f}"
    )


def component_description(
    component,
    role
):

    d = descriptor_lookup[
        component
    ]

    properties = [

        descriptor_phrase(
            name,
            d[
                name
            ]
        )

        for name in (
            FINAL_DESCRIPTOR_NAMES
        )
    ]


    return (

        f"The {role} building block "
        f"{component} has "
        +
        ", ".join(
            properties
        )
        +
        "."
    )


# ============================================================
# 20. CHEMICAL LANGUAGE GENERATION
# ============================================================
#
# Deliberately rule-based and deterministic.
#
# No experimental response is used.
#
# ============================================================

language_rows = []


for _, row in (
    comp.iterrows()
):

    canonical = row[
        "Canonical_ID"
    ]

    copolymer = row[
        "Copolymer_Name"
    ]


    hydrophilic = row[
        "mono_1"
    ]

    hydrophobic = row[
        "mono_2"
    ]

    crosslinker = row[
        "mono_3"
    ]


    f_hphil = float(
        row[
            "comp_1"
        ]
    )

    f_hphob = float(
        row[
            "comp_2"
        ]
    )

    f_cross = float(
        row[
            "comp_3"
        ]
    )


    # --------------------------------------------------------
    # Role sentence
    # --------------------------------------------------------

    role_sentence = (

        f"{copolymer} is a crosslinked copolymer "
        f"constructed from {hydrophilic} as the "
        f"hydrophilic building block, "
        f"{hydrophobic} as the hydrophobic "
        f"building block, and {crosslinker} "
        f"as the crosslinker."
    )


    # --------------------------------------------------------
    # Composition sentence
    # --------------------------------------------------------

    composition_sentence = (

        f"The composition fractions are "
        f"{fmt_fraction(f_hphil)} hydrophilic, "
        f"{fmt_fraction(f_hphob)} hydrophobic, "
        f"and {fmt_fraction(f_cross)} crosslinker."
    )


    # --------------------------------------------------------
    # Physicochemical descriptions
    # --------------------------------------------------------

    hydrophilic_sentence = (
        component_description(
            hydrophilic,
            "hydrophilic"
        )
    )


    hydrophobic_sentence = (
        component_description(
            hydrophobic,
            "hydrophobic"
        )
    )


    crosslinker_sentence = (
        component_description(
            crosslinker,
            "crosslinking"
        )
    )


    # --------------------------------------------------------
    # Explicit contrast sentence
    # --------------------------------------------------------

    d_hphil = (
        descriptor_lookup[
            hydrophilic
        ]
    )

    d_hphob = (
        descriptor_lookup[
            hydrophobic
        ]
    )


    delta_logp = (

        float(
            d_hphob[
                "cLogP"
            ]
        )

        -

        float(
            d_hphil[
                "cLogP"
            ]
        )
    )


    delta_tpsa = (

        float(
            d_hphil[
                "TPSA"
            ]
        )

        -

        float(
            d_hphob[
                "TPSA"
            ]
        )
    )


    if delta_logp > 0:

        logp_relation = (

            f"{hydrophobic} is more "
            f"lipophilic than {hydrophilic}"
        )

    elif delta_logp < 0:

        logp_relation = (

            f"{hydrophilic} is more "
            f"lipophilic than {hydrophobic}"
        )

    else:

        logp_relation = (

            f"{hydrophilic} and "
            f"{hydrophobic} have similar "
            f"calculated lipophilicity"
        )


    if delta_tpsa > 0:

        polar_relation = (

            f"{hydrophilic} has a larger "
            f"polar surface area than "
            f"{hydrophobic}"
        )

    elif delta_tpsa < 0:

        polar_relation = (

            f"{hydrophobic} has a larger "
            f"polar surface area than "
            f"{hydrophilic}"
        )

    else:

        polar_relation = (

            f"{hydrophilic} and "
            f"{hydrophobic} have similar "
            f"polar surface area"
        )


    contrast_sentence = (

        f"In the building-block contrast, "
        f"{logp_relation}, while "
        f"{polar_relation}."
    )


    # --------------------------------------------------------
    # Final sentence
    # --------------------------------------------------------

    full_sentence = " ".join(

        [

            role_sentence,

            composition_sentence,

            hydrophilic_sentence,

            hydrophobic_sentence,

            crosslinker_sentence,

            contrast_sentence,
        ]
    )


    language_rows.append(

        {

            "Canonical_ID":
                canonical,

            "Copolymer_Name":
                copolymer,

            "Hydrophilic_component":
                hydrophilic,

            "Hydrophobic_component":
                hydrophobic,

            "Crosslinker_component":
                crosslinker,

            "Hydrophilic_fraction":
                f_hphil,

            "Hydrophobic_fraction":
                f_hphob,

            "Crosslinker_fraction":
                f_cross,

            "Role_sentence":
                role_sentence,

            "Composition_sentence":
                composition_sentence,

            "Hydrophilic_sentence":
                hydrophilic_sentence,

            "Hydrophobic_sentence":
                hydrophobic_sentence,

            "Crosslinker_sentence":
                crosslinker_sentence,

            "Contrast_sentence":
                contrast_sentence,

            "Chemical_Language":
                full_sentence,
        }
    )


language_df = pd.DataFrame(
    language_rows
)


LANGUAGE_PATH = (

    OUTPUT_DIR
    /
    "04_Chemical_Language_Sentences.csv"
)


language_df.to_csv(

    LANGUAGE_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 21. SENTENCE EMBEDDING
# ============================================================

print(
    "\nLoading language model:"
)

print(
    LANGUAGE_MODEL_NAME
)


model = SentenceTransformer(
    LANGUAGE_MODEL_NAME
)


sentences = (

    language_df[
        "Chemical_Language"
    ]

    .fillna("")

    .astype(str)

    .tolist()
)


embedding = model.encode(

    sentences,

    normalize_embeddings=True,

    show_progress_bar=True
)


embedding = np.asarray(
    embedding,
    dtype=float
)


if embedding.shape[1] != 384:

    print(

        "WARNING: expected 384 embedding "
        f"dimensions but obtained "
        f"{embedding.shape[1]}"
    )


embedding_columns = [

    f"ChemLang_{i:03d}"

    for i in range(
        embedding.shape[1]
    )
]


embedding_df = pd.DataFrame(

    embedding,

    columns=embedding_columns
)


embedding_df.insert(

    0,

    "Copolymer_Name",

    language_df[
        "Copolymer_Name"
    ].values
)


embedding_df.insert(

    0,

    "Canonical_ID",

    language_df[
        "Canonical_ID"
    ].values
)


EMBEDDING_PATH = (

    OUTPUT_DIR
    /
    "05_Chemical_Language_Embedding_384D.csv"
)


embedding_df.to_csv(

    EMBEDDING_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 22. AUDIT TABLE
# ============================================================

audit = comp[
    [
        "Canonical_ID",
        "Copolymer_Name",
        "mono_1",
        "mono_2",
        "mono_3",
        "comp_1",
        "comp_2",
        "comp_3",
        "Composition_sum",
    ]
].copy()


audit[
    "All_components_have_SMILES"
] = (

    audit.apply(

        lambda row:

        (
            row[
                "mono_1"
            ]
            in smiles_components
        )

        and

        (
            row[
                "mono_2"
            ]
            in smiles_components
        )

        and

        (
            row[
                "mono_3"
            ]
            in smiles_components
        ),

        axis=1
    )
)


audit[
    "Composition_sum_close_to_0.97"
] = (

    np.isclose(

        audit[
            "Composition_sum"
        ],

        0.97,

        atol=0.02
    )
)


audit[
    "Chemical_Language_present"
] = (

    language_df[
        "Chemical_Language"
    ]
    .str.len()
    >
    0
)


AUDIT_PATH = (

    OUTPUT_DIR
    /
    "06_Chemical_representation_audit.csv"
)


audit.to_csv(

    AUDIT_PATH,

    index=False,

    encoding="utf-8-sig"
)


# ============================================================
# 23. SOURCE DATA WORKBOOK
# ============================================================

SOURCE_PATH = (

    OUTPUT_DIR
    /
    "07_Chemical_representation_source_data.xlsx"
)


with pd.ExcelWriter(

    SOURCE_PATH,

    engine="openpyxl"

) as writer:


    comp.to_excel(

        writer,

        sheet_name="Composition",

        index=False
    )


    component_desc.to_excel(

        writer,

        sheet_name="Component_descriptors",

        index=False
    )


    role_df.to_excel(

        writer,

        sheet_name="RoleSeparated_RDKit",

        index=False
    )


    rdkit_final.to_excel(

        writer,

        sheet_name="RDKit_Final6D",

        index=False
    )


    language_df.to_excel(

        writer,

        sheet_name="Chemical_Language",

        index=False
    )


    audit.to_excel(

        writer,

        sheet_name="Audit",

        index=False
    )


# ============================================================
# 24. METADATA
# ============================================================

try:

    import rdkit

    rdkit_version = (
        rdkit.__version__
    )

except Exception:

    rdkit_version = (
        "unknown"
    )


metadata = {

    "pipeline":
        "04_Chemical_representation",

    "purpose":
        (
            "Generate Y-independent chemical "
            "representations from composition "
            "and molecular structure only."
        ),

    "inputs": [

        COMPOSITION_FILE.name,

        SMILES_FILE.name,
    ],

    "expected_materials":
        EXPECTED_N,

    "RDKit_version":
        rdkit_version,

    "RDKit_final_descriptors":
        FINAL_DESCRIPTOR_NAMES,

    "RDKit_final_dimension":
        len(
            FINAL_DESCRIPTOR_NAMES
        ),

    "role_separation": [

        "hydrophilic",

        "hydrophobic",

        "crosslinker",
    ],

    "language_model":
        LANGUAGE_MODEL_NAME,

    "embedding_normalized":
        True,

    "chemical_language_sources": [

        "building-block identity",

        "building-block role",

        "composition fraction",

        "RDKit physicochemical descriptors",

        "hydrophilic-versus-hydrophobic "
        "building-block contrast",
    ],

    "TD_NMR_used":
        False,

    "Dynamic_State_Language_used":
        False,

    "Solution_NMR_used":
        False,

    "experimental_Y_used":
        False,

    "leakage_control":
        (
            "Chemical representation is generated "
            "without TD-NMR or Solution-NMR response "
            "information."
        ),
}


METADATA_PATH = (

    OUTPUT_DIR
    /
    "08_Metadata.json"
)


with open(

    METADATA_PATH,

    "w",

    encoding="utf-8"

) as f:


    json.dump(

        metadata,

        f,

        indent=2,

        ensure_ascii=False
    )


# ============================================================
# 25. QC
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "04 CHEMICAL REPRESENTATION — QC"
)


print(
    "=" * 80
)


print(
    "\nMaterials:",
    len(
        comp
    )
)


print(
    "Unique Canonical_ID:",
    comp[
        "Canonical_ID"
    ].nunique()
)


print(
    "Components:",
    len(
        component_desc
    )
)


print(
    "RDKit dimensions:",
    len(
        FINAL_DESCRIPTOR_NAMES
    )
)


print(
    "Chemical Language embedding shape:",
    embedding.shape
)


print(
    "\nComposition sums:"
)


print(

    comp[
        "Composition_sum"
    ]
    .describe()
)


# ============================================================
# 26. STRICT QC
# ============================================================

if len(
    comp
) != EXPECTED_N:

    print(

        f"\nWARNING: expected "
        f"{EXPECTED_N} materials "
        f"but obtained {len(comp)}."
    )


if (
    comp[
        "Canonical_ID"
    ]
    .duplicated()
    .any()
):

    duplicates = (

        comp.loc[
            comp[
                "Canonical_ID"
            ]
            .duplicated(
                keep=False
            )
        ]
    )

    raise ValueError(

        "Duplicate Canonical_ID detected:\n"

        + duplicates[
            [
                "Canonical_ID",
                "Copolymer_Name",
            ]
        ]
        .to_string(
            index=False
        )
    )


if not audit[
    "All_components_have_SMILES"
].all():

    raise ValueError(
        "Some components do not have SMILES."
    )


if rdkit_final[
    FINAL_DESCRIPTOR_NAMES
].isna().any().any():

    raise ValueError(
        "NaN detected in final RDKit representation."
    )


if np.isnan(
    embedding
).any():

    raise ValueError(
        "NaN detected in Chemical Language embedding."
    )


# ============================================================
# 27. PRINT EXAMPLE CHEMICAL LANGUAGE
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "EXAMPLE CHEMICAL LANGUAGE"
)


print(
    "=" * 80
)


for i in range(
    min(
        3,
        len(
            language_df
        )
    )
):

    print(
        "\n",
        language_df.loc[
            i,
            "Canonical_ID"
        ]
    )

    print(
        language_df.loc[
            i,
            "Chemical_Language"
        ]
    )


# ============================================================
# 28. ZIP
# ============================================================

output_files = [

    COMPONENT_PATH,

    ROLE_PATH,

    RDKIT_FINAL_PATH,

    LANGUAGE_PATH,

    EMBEDDING_PATH,

    AUDIT_PATH,

    SOURCE_PATH,

    METADATA_PATH,
]


with zipfile.ZipFile(

    ZIP_PATH,

    "w",

    compression=zipfile.ZIP_DEFLATED

) as z:


    for path in output_files:

        z.write(

            path,

            arcname=path.name
        )


# ============================================================
# 29. FINAL REPORT
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "GENERATED FILES"
)


print(
    "=" * 80
)


for path in output_files:

    print(
        " -",
        path.name
    )


print(
    "\nZIP:"
)


print(
    ZIP_PATH
)


print(
    "\nRepresentation architecture:"
)


print(
    "Composition / SMILES"
)


print(
    "       ↓"
)


print(
    "RDKit physicochemical descriptors"
)


print(
    "       ↓"
)


print(
    "Role-separated chemical representation"
)


print(
    "       ├── RDKit Numeric 6D"
)


print(
    "       └── Chemical Language"
)


print(
    "                ↓"
)


print(
    "       MiniLM 384D embedding"
)


print(
    "\nNo experimental Y was used."
)


# ============================================================
# 30. DOWNLOAD
# ============================================================

try:

    from google.colab import files

    files.download(
        str(
            ZIP_PATH
        )
    )

except ImportError:

    pass

[INSTALL] rdkit
Upload TWO files:
1) copolymer_composition.csv
2) component_smiles.csv


Saving component_smiles.csv to component_smiles.csv
Saving copolymer_composition.csv to copolymer_composition.csv

Composition: copolymer_composition.csv
SMILES: component_smiles.csv
[INFO] Loaded copolymer_composition.csv with encoding=utf-8-sig
[INFO] Loaded component_smiles.csv with encoding=utf-8-sig

Composition columns:
['Copolymer_Name', 'mono_1', 'mono_2', 'mono_3', 'comp_1', 'comp_2', 'comp_3']

SMILES columns:
['Component', 'smiles']

Loading language model:
sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]


04 CHEMICAL REPRESENTATION — QC

Materials: 43
Unique Canonical_ID: 43
Components: 13
RDKit dimensions: 6
Chemical Language embedding shape: (43, 384)

Composition sums:
count    4.300000e+01
mean     9.700000e-01
std      6.880227e-16
min      9.700000e-01
25%      9.700000e-01
50%      9.700000e-01
75%      9.700000e-01
max      9.700000e-01
Name: Composition_sum, dtype: float64

EXAMPLE CHEMICAL LANGUAGE

 AMPS_HMA_TECL
AMPS_HMA_TECL is a crosslinked copolymer constructed from AMPS as the hydrophilic building block, HMA as the hydrophobic building block, and TECL as the crosslinker. The composition fractions are 0.435 hydrophilic, 0.435 hydrophobic, and 0.100 crosslinker. The hydrophilic building block AMPS has cLogP -3.38, TPSA 86.3 square angstrom, 5 hydrogen-bond donor/acceptor sites in total, 4 rotatable bonds, formal ionic character, heteroatom fraction 0.50. The hydrophobic building block HMA has cLogP 2.69, TPSA 26.3 square angstrom, 2 hydrogen-bond donor/acceptor sites in t

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>